# 31. 제목 계층 문맥 반환 정책·예산 비교

목적: 30번 C의 0점을 검색 실패와 구분하고, 같은 제목 계층 검색 결과를 **어떻게 반환할지** 비교한다.

- A: 제목 경계 청크 그대로. B: 토큰 부모 전체. C-full: 기존 실제 제목 부모 전체.
- C-seed-first: 상위 5개 검색 조각을 먼저 보존하고, 남는 공간에서 그 조각의 **직접 제목 문맥**으로 확장한다. 큰 상위 장 전체로 무조건 올라가지 않는다.
- 공통 조건: 같은 5기업·2024 사업보고서·30개 개발 질문, E5 + Kiwi BM25 RRF 5:5. C-full/C-seed-first의 검색 순위는 완전히 같다.
- 제한 없음이 기본 해석 대상. 2,400 / 8,000 / 16,000 / 32,000 E5 토큰은 민감도 비교용 가정이며 운영 LLM의 확정 한도가 아니다.
- E5는 **GPU 0, FP32**. 다른 GPU 모델을 중단·재로딩하지 않는다. LLM 답변 생성·API 과금·최종 pipeline 구현은 하지 않는다.
- 근거 인용문은 평가에만 사용하며, 반환 정책에 정답·질문 의도를 전달하지 않는다. 이전 질문을 재사용한 개발 비교로 독립 holdout이 아니다.

Codex가 skn25에서 전체 실행하고 저장한다. 30번 notebook과 운영 src는 변경하지 않는다.

## 1. 검증된 검색 코드·고정 질문 재사용
30번 코드만 해시 검증 후 재사용한다. 저장 출력이나 GPU/CPU 모델 실행 셀은 자동 실행하지 않는다.

In [1]:
from pathlib import Path
import hashlib, json
from functools import lru_cache

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'src/dart/client.py').is_file())
nb30 = json.loads((PROJECT_ROOT / 'notebooks/v1/30_parent_child_chunking_comparison.ipynb').read_text())
reused_cells = {c['id']: ''.join(c['source']) for c in nb30['cells'] if c['cell_type'] == 'code'}
source30_sha = hashlib.sha256('\n'.join(c['id']+'\n'+''.join(c['source'])
    for c in nb30['cells'] if c['cell_type']=='code').encode()).hexdigest()
if source30_sha != 'd38daeb5664a0ba916fa908187de7abf39dbf57311771e08b2005065ffc6d952':
    raise RuntimeError('30번의 검토된 코드가 변경됐습니다. 재검토 후 실행하세요.')

def reuse_cell(cell_id):
    source = reused_cells[cell_id]
    # 원래 setup의 CPU 안내만 생략한다. 실제 GPU 로드는 이번 notebook에서 명시한다.
    if cell_id == 'v30-02':
        source = '\n'.join(line for line in source.splitlines() if not line.startswith("print('skn25 |"))
    exec(compile(source, f'<approved-30:{cell_id}>', 'exec'), globals())

reuse_cell('v30-02')
reuse_cell('v30-04')
BUDGETS = (None, 2400, 8000, 16000, 32000)
POLICIES = {'A':'title_boundary', 'B':'parent_child',
            'C-full':'heading_parent_child', 'C-seed-first':'heading_parent_child'}
print('skn25 | E5 cuda:0 FP32 | 30개 개발 질문 | LLM 호출 없음', flush=True)


/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,id,company,dataset_group,question,reference_ids
0,S01,삼성전자,single_25,삼성전자가 주식을 일반에 공개한 시점은 언제인가?,[S01]
1,S02,삼성전자,single_25,삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?,[S02]
2,S03,삼성전자,single_25,삼성전자 본사의 소재지는 어디인가?,[S03]
3,S04,삼성전자,single_25,삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?,[S04]
4,S05,삼성전자,single_25,삼성전자 전장 사업에 해당하는 하만의 주요 제품은?,[S05]
5,R01,리노공업,single_25,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01]
6,R02,리노공업,single_25,리노공업의 해외 거래처 관리 방식은 무엇인가?,[R02]
7,R03,리노공업,single_25,리노공업의 공정가치로 측정되는 금융상품 관련 공시 근거를 찾아줘.,[R03]
8,R04,리노공업,single_25,리노공업 퇴직급여 표에서 확정급여채무와 함께 어떤 적립자산 항목을 확인할 수 있나?,[R04]
9,R05,리노공업,single_25,리노공업이 시장 환경에서 경계하는 원자재·에너지 관련 요인은?,[R05]


고정 질문·근거 SHA256: a8bbc6202b3b0956c7519660a6e760eb1dab9be1d808118bdba1e7557c3c6ebe
skn25 | E5 cuda:0 FP32 | 30개 개발 질문 | LLM 호출 없음


## 2. 동일 접수번호의 XML·본문 준비
원문과 고정 인용문 보존 gate를 유지한다. 네트워크 오류를 데이터 부재로 바꾸지 않는다.

In [2]:
reuse_cell('v30-06')
source_sha256 = {company: hashlib.sha256(data).hexdigest() for company, data in source_xml.items()}
print('XML 준비:', list(source_xml), flush=True)


삼성전자 원문 준비 시작 | 접수번호 20250311001085


삼성전자 준비 완료 | 44,847자


리노공업 원문 준비 시작 | 접수번호 20250318000229


리노공업 준비 완료 | 159,669자


NAVER 원문 준비 시작 | 접수번호 20250318000645


...

LG생활건강 준비 완료 | 426,665자


HDC 원문 준비 시작 | 접수번호 20250318001151


HDC 준비 완료 | 62,037자


,company,corp_code,business_year,receipt,xml_MB,text_chars,text_utf8_MB,source_sha256,download_extract_seconds,source_url
0,삼성전자,00126380,2024,20250311001085,6.006890,44847,0.079431,b0682fd46de50ecc41f074bb43cb58cf574dd175034371...,0.460372,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,리노공업,00369657,2024,20250318000229,1.486798,159669,0.298023,a23af2dbcc2f4de4be59c5925ed45fceae84c23f4e78ff...,0.683351,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,NAVER,00266961,2024,20250318000645,6.900676,639240,1.079369,b1478f612c3ff07373d14fe9c7638936c77fcea2130aa4...,3.419784,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,LG생활건강,00356370,2024,20250317000790,4.573826,426665,0.764530,9ccc7d18e1ba66ce3982f41899d22d1dcb7ae8dfaf784f...,2.799488,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,HDC,00164636,2024,20250318001151,6.137928,62037,0.108997,53b41f2833b68f06bfea13a07abed1a7cdb3caf53c44e6...,0.478693,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


XML 준비: ['삼성전자', '리노공업', 'NAVER', 'LG생활건강', 'HDC']


## 3. 동일 청킹 후보와 실제 제목 소유 관계 준비
A/B/C 검색 단위는 30번 그대로다. C-seed-first만 반환 정책을 바꾸고 임베딩을 추가하지 않는다.

In [3]:
for cell_id in ('v30-08', 'v30-heading-tree', 'v30-heading-candidate', 'v30-10'):
    reuse_cell(cell_id)
count_tokens = lru_cache(maxsize=2048)(count_tokens)

# 기존 C 조각이 어느 직접 제목에서 만들어졌는지 복원한다.
# 중복 본문이라 유일하게 식별되지 않으면 임의 제목을 골라 확장하지 않고 검색 조각을 유지한다.
owners_by_company, own_contexts_by_company, ownership_rows = {}, {}, []
for company in source_texts:
    records, _ = heading_tree(source_xml[company], source_texts[company])
    key = ('heading_parent_child', company)
    entries = [(r['id'], compact(r['own_text']),
                r['id'] if r['children'] else (r['parent_id'] or r['id'])) for r in records]
    owners = []
    for piece, parent_id in zip(texts_by_key[key], payload_ids_by_key[key]):
        needle = compact(piece)
        matches = [rid for rid, own, payload in entries if payload == parent_id and needle in own]
        owners.append(matches[0] if len(matches) == 1 else None)
    owners_by_company[company] = owners
    own_contexts_by_company[company] = {r['id']: r['context'] for r in records}
    ownership_rows.append({'company':company, 'search_chunks':len(owners),
                           'unresolved_owner_chunks':sum(o is None for o in owners)})
display(pd.DataFrame(ownership_rows))


원문 제목 계층·형제 구분·부모 본문 보존·실제 반환 정책 검사 완료


title_boundary | 삼성전자 | 검색 청크 82 | 반환 단위 82


title_boundary | 리노공업 | 검색 청크 333 | 반환 단위 333


title_boundary | NAVER | 검색 청크 1168 | 반환 단위 1168


title_boundary | LG생활건강 | 검색 청크 824 | 반환 단위 824


...

,method,company,search_chunks,return_units,max_e5_tokens,chunking_seconds,search_text_MiB,context_text_MiB
0,title_boundary,삼성전자,82,82,403,0.248956,0.082886,0.082886
1,title_boundary,리노공업,333,333,404,0.525678,0.309031,0.309031
2,title_boundary,NAVER,1168,1168,404,2.749327,1.146790,1.146790
3,title_boundary,LG생활건강,824,824,404,1.701303,0.811188,0.811188
4,title_boundary,HDC,123,123,404,0.237674,0.115170,0.115170
5,parent_child,삼성전자,91,32,403,0.214158,0.084018,0.080358
6,parent_child,리노공업,349,146,404,0.804464,0.309011,0.299274
7,parent_child,NAVER,1293,418,404,4.264196,1.153458,1.096161
8,parent_child,LG생활건강,902,325,404,2.897689,0.815345,0.773008
9,parent_child,HDC,134,46,399,0.383090,0.115545,0.110558


,method,id,company,source_exists,search_gold_chunks,return_gold_units
0,title_boundary,S01,삼성전자,True,1,1
1,title_boundary,S02,삼성전자,True,1,1
2,title_boundary,S03,삼성전자,True,4,4
3,title_boundary,S04,삼성전자,True,2,2
4,title_boundary,S05,삼성전자,True,1,1
...,...,...,...,...,...,...
70,heading_parent_child,H01,HDC,True,1,1
71,heading_parent_child,H02,HDC,True,2,1
72,heading_parent_child,H03,HDC,True,1,1
73,heading_parent_child,H04,HDC,True,1,1


,company,heading_nodes,structural_edges,oversized_heading_bodies,preamble_search_pieces
0,삼성전자,16,12,10,1
1,리노공업,98,82,41,1
2,NAVER,135,131,94,1
3,LG생활건강,154,149,93,1
4,HDC,14,10,9,1


,company,id,level,title,parent_id,context_e5_tokens
0,삼성전자,heading:0,1,목 차,NaN,3
1,삼성전자,heading:1,1,【 대표이사 등의 확인 】,NaN,18
2,삼성전자,heading:2,1,I. 회사의 개요,NaN,9732
3,삼성전자,heading:3,2,1. 회사의 개요,heading:2,2750
4,삼성전자,heading:4,2,2. 회사의 연혁,heading:2,2082
...,...,...,...,...,...,...
412,HDC,heading:9,2,1. 사업의 개요,heading:8,1086
413,HDC,heading:10,2,2. 주요 제품 및 서비스,heading:8,2602
414,HDC,heading:11,2,3. 원재료 및 생산설비,heading:8,10214
415,HDC,heading:12,2,4. 매출 및 수주상황,heading:8,9108


,company,search_chunks,unresolved_owner_chunks
0,삼성전자,79,1
1,리노공업,316,2
2,NAVER,1155,2
3,LG생활건강,805,1
4,HDC,120,1


## 4. BM25와 GPU 0 E5 색인
E5 접두어·FP32·정규화는 유지하고 GPU 장치만 지정한다. 후보 사이 동일 텍스트 벡터는 한 번만 계산한다.

In [4]:
if os.getenv('CUDA_VISIBLE_DEVICES') not in (None, '', '0', '0,1,2'):
    raise RuntimeError('CUDA_VISIBLE_DEVICES가 GPU 0을 재매핑했을 수 있습니다. 물리 GPU를 먼저 확인하세요.')
if not torch.cuda.is_available():
    raise RuntimeError('GPU 0을 사용할 수 없습니다. CPU로 조용히 대체하지 않습니다.')
DEVICE = torch.device('cuda:0')
free_bytes, total_bytes = torch.cuda.mem_get_info(DEVICE)
if free_bytes < 2 * 1024**3:
    raise RuntimeError('GPU 0 여유 VRAM 2GiB 미만. 다른 모델을 내리지 않고 실행을 중단합니다.')
gpu_initial_free_MiB = free_bytes / 1024**2
torch.cuda.reset_peak_memory_stats(DEVICE)
started = perf_counter()
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                 use_safetensors=True).to(DEVICE).eval()
torch.cuda.synchronize(DEVICE)
model_load_seconds = perf_counter() - started
parameter_MiB = sum(p.numel()*p.element_size() for p in model.parameters())/1024**2

def encode_texts(texts, kind, batch_size=16):
    rows = []
    for start in range(0, len(texts), batch_size):
        batch = [f'{kind}: {text}' for text in texts[start:start + batch_size]]
        tokens = tokenizer(batch, padding=True, truncation=False, return_tensors='pt')
        if tokens['input_ids'].shape[1] > 512:
            raise RuntimeError('E5 입력 초과. 조용히 자르지 않습니다.')
        tokens = {name: value.to(DEVICE) for name, value in tokens.items()}
        with torch.inference_mode():
            hidden = model(**tokens).last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            rows.append(F.normalize(pooled, p=2, dim=1).cpu().numpy().astype(np.float32))
    return np.vstack(rows)

print('E5 장치:', next(model.parameters()).device, torch.cuda.get_device_name(0), flush=True)
reuse_cell('v30-12')
embedding_cache, dense_index, e5_rows = {}, {}, []
for key, texts in texts_by_key.items():
    started = perf_counter()
    missing = [t for t in dict.fromkeys(texts) if t not in embedding_cache]
    for start in range(0, len(missing), 64):
        batch = missing[start:start+64]
        embedding_cache.update(zip(batch, encode_texts(batch, 'passage')))
        print(f'{key}: 신규 {min(start+64,len(missing))}/{len(missing)}', flush=True)
    dense_index[key] = np.stack([embedding_cache[t] for t in texts])
    e5_rows.append({'method':key[0], 'company':key[1], 'new_embedding_texts':len(missing),
                   'reused_records':len(texts)-len(missing), 'e5_build_seconds':perf_counter()-started,
                   'vectors_MiB':dense_index[key].nbytes/1024**2})
display(pd.DataFrame(e5_rows))


E5 장치: cuda:0 NVIDIA GeForce RTX 3090


,method,company,bm25_build_seconds,sqlite_MiB,empty_bm25_chunks
0,title_boundary,삼성전자,1.806644,0.140625,0
1,title_boundary,리노공업,1.870240,0.421875,0
2,title_boundary,NAVER,5.748509,1.476562,1
3,title_boundary,LG생활건강,4.692846,1.031250,0
4,title_boundary,HDC,0.642436,0.179688,0
5,parent_child,삼성전자,0.472539,0.140625,0
6,parent_child,리노공업,1.970275,0.417969,0
7,parent_child,NAVER,5.774736,1.476562,2
8,parent_child,LG생활건강,4.688148,1.027344,0
9,parent_child,HDC,0.644519,0.183594,0


('title_boundary', '삼성전자'): 신규 64/82


('title_boundary', '삼성전자'): 신규 82/82


('title_boundary', '리노공업'): 신규 64/331


...

('heading_parent_child', '리노공업'): 신규 7/7


('heading_parent_child', 'NAVER'): 신규 4/4


('heading_parent_child', 'LG생활건강'): 신규 9/9


('heading_parent_child', 'HDC'): 신규 1/1


,method,company,new_embedding_texts,reused_records,e5_build_seconds,vectors_MiB
0,title_boundary,삼성전자,82,0,0.543685,0.120117
1,title_boundary,리노공업,331,2,0.877144,0.487793
2,title_boundary,NAVER,1156,12,3.051405,1.710938
3,title_boundary,LG생활건강,809,15,2.122995,1.207031
4,title_boundary,HDC,120,3,0.332524,0.180176
5,parent_child,삼성전자,32,59,0.072985,0.133301
6,parent_child,리노공업,84,265,0.218231,0.511230
7,parent_child,NAVER,668,625,1.736608,1.894043
8,parent_child,LG생활건강,395,507,0.988677,1.321289
9,parent_child,HDC,73,61,0.192149,0.196289


## 5. 반환 정책 — 정답을 사용하지 않는다

기존 A/B/C-full: 기존 단위를 순서대로 통째로 담고 넘치면 중단. 제한 없음이면 전부 반환.

C-seed-first: 작은 검색 조각을 순위대로 먼저 담고, 이미 담은 조각을 잃지 않는 조건에서 그 조각의 직접 제목 문맥으로 확장. 같은 직접 제목 조각은 한 문맥으로 병합한다. 예산에 안 맞으면 작은 조각을 유지한다. 제목 관계가 모호하면 추측하지 않는다. 제한 없음에서도 직접 제목 문맥까지만 확장하며, 무조건 상위 장 전체로 올라가지 않는다.

이것은 LlamaIndex AutoMergingRetriever의 공식 임계값 알고리즘이 아니라 FinSight 실험 후보다. 검색 순위·인용문 정답·질문은 반환 정책의 판단에 사용하지 않는다.

In [5]:
reuse_cell('v30-16')

def ranked_prefix(units, budget):
    if budget is None:
        return units
    return pack_contexts(units, budget=budget)

def seed_first_contexts(seeds, owners, owner_contexts, budget, token_counter=count_tokens):
    chosen = []
    for rank, (seed, owner) in enumerate(zip(seeds, owners)):
        if owner is not None and any(item['owner'] == owner and item['text'] == seed for item in chosen):
            continue
        proposal = chosen + [{'rank':rank, 'owner':owner, 'text':seed}]
        if budget is None or token_counter('\n\n'.join(i['text'] for i in proposal)) <= budget:
            chosen = proposal
        else:
            break
    # 이미 담은 검색 조각의 근거를 희생하지 않고, 같은 제목의 조각들만 문맥으로 병합한다.
    for owner in dict.fromkeys(i['owner'] for i in chosen if i['owner'] is not None):
        group = [i for i in chosen if i['owner'] == owner]
        expanded = owner_contexts[owner]
        if not all(compact(i['text']) in compact(expanded) for i in group):
            raise RuntimeError('확장 문맥이 검색 조각을 보존하지 않습니다.')
        merged = {'rank':min(i['rank'] for i in group), 'owner':owner, 'text':expanded}
        proposal = sorted([i for i in chosen if i['owner'] != owner] + [merged], key=lambda i:i['rank'])
        if budget is None or token_counter('\n\n'.join(i['text'] for i in proposal)) <= budget:
            chosen = proposal
    return [i['text'] for i in chosen]

# 최소 정책 검사: 큰 부모가 있어도 검색 조각을 보존, 제목별 병합, 제한 없는 확장, 모호 제목 fallback.
probe_seeds, probe_owners = ['ONE', 'TWO', 'THREE'], ['h1', 'h1', 'h2']
probe_contexts = {'h1':'ONE TWO EXTRA', 'h2':'THREE ' + 'EXTRA '*20}
probe_budget = len('\n\n'.join(probe_seeds))
probe_small = seed_first_contexts(probe_seeds, probe_owners, probe_contexts, probe_budget, len)
assert all(s in '\n\n'.join(probe_small) for s in probe_seeds)
assert len('\n\n'.join(probe_small)) <= probe_budget
probe_full = seed_first_contexts(probe_seeds, probe_owners, probe_contexts, None, len)
assert probe_full == [probe_contexts['h1'], probe_contexts['h2']]
assert seed_first_contexts(['ONE'], [None], {}, 10, len) == ['ONE']
assert seed_first_contexts(['ONE'], [None], {}, 0, len) == []
assert seed_first_contexts(['SHARED','SHARED'], ['a','b'],
    {'a':'SHARED A','b':'SHARED B'}, None, len) == ['SHARED A','SHARED B']
assert seed_first_contexts(['SHARED','SHARED'], [None,None], {}, None, len) == ['SHARED','SHARED']
assert seed_first_contexts(['LONG','X'], [None,None], {}, 1, len) == []
assert ranked_prefix(['ONE'], None) == ['ONE']
print('반환 정책의 검색 조각 보존·제목 병합·예산 검사 통과')


부모 연결 반환·중복 제거·공통 예산·고정 근거 Recall 검사 완료
반환 정책의 검색 조각 보존·제목 병합·예산 검사 통과


## 6. 검색은 한 번, 반환 정책·예산만 비교
각 C 질문의 같은 top-5를 C-full/C-seed-first에 재사용한다. E5/CPU 수치 차이가 있을 수 있어 30번의 저장 숫자를 정답으로 강제하지 않는다.

In [6]:
rows, search_rows, failure_rows = [], [], []
for number, case in enumerate(qcases, 1):
    print(f"[{number}/30] {case['id']} {case['company']} 검색", flush=True)
    started = perf_counter()
    query_vector = encode_texts([case['question']], 'query')[0]
    query_ms = (perf_counter()-started)*1000
    orders = {}
    for method in METHODS:
        key = (method, case['company'])
        started = perf_counter()
        lex = bm25_order(key, case['question'])
        dense = np.argsort(-(dense_index[key] @ query_vector), kind='stable')
        orders[method] = weighted_rrf(dense, lex, len(texts_by_key[key]), VECTOR_WEIGHT)
        leaves = [texts_by_key[key][int(i)] for i in orders[method][:EVAL_K]]
        rank = next((rank for rank, i in enumerate(orders[method], 1)
                     if evidence_recall([texts_by_key[key][int(i)]], case['reference_ids']) > 0), None)
        search_rows.append({'id':case['id'], 'method':method, 'company':case['company'],
            'leaf_hit':rank is not None and rank <= EVAL_K,
            'leaf_mrr':1/rank if rank is not None and rank <= EVAL_K else 0,
            'leaf_recall':evidence_recall(leaves, case['reference_ids']),
            'search_ms':query_ms+(perf_counter()-started)*1000})
    for policy, method in POLICIES.items():
        key = (method, case['company'])
        order = orders[method]
        seeds = [texts_by_key[key][int(i)] for i in order[:EVAL_K]]
        owners = [owners_by_company[case['company']][int(i)] for i in order[:EVAL_K]] if policy == 'C-seed-first' else []
        for budget in BUDGETS:
            started = perf_counter()
            if policy == 'C-seed-first':
                contexts = seed_first_contexts(seeds, owners, own_contexts_by_company[case['company']], budget)
            else:
                contexts = ranked_prefix(returned_contexts(key, order), budget)
            tokens = count_tokens('\n\n'.join(contexts))
            assert budget is None or tokens <= budget
            if policy == 'C-seed-first' and (budget is None or count_tokens('\n\n'.join(seeds)) <= budget):
                assert all(any(compact(seed) in compact(ctx) for ctx in contexts) for seed in seeds)
            recall = evidence_recall(contexts, case['reference_ids'])
            rows.append({'id':case['id'], 'company':case['company'], 'dataset_group':case['dataset_group'],
                'policy':policy, 'budget':'unlimited' if budget is None else str(budget),
                'hit':recall > 0, 'recall':recall, 'all_evidence':recall == 1,
                'context_tokens':tokens, 'context_units':len(contexts),
                'covered_seed_ranks':[i+1 for i, seed in enumerate(seeds)
                    if any(compact(seed) in compact(ctx) for ctx in contexts)],
                'context_ms':(perf_counter()-started)*1000})
            if recall < 1 and (budget is None or budget == 8000):
                failure_rows.append({'id':case['id'], 'company':case['company'], 'policy':policy,
                    'budget':'unlimited' if budget is None else str(budget), 'question':case['question'],
                    'missing_ids':[r for r in case['reference_ids'] if evidence_recall(contexts,[r]) == 0],
                    'context_tokens':tokens, 'first_context_preview':contexts[0][:200] if contexts else ''})
    print('  4개 정책 × 5개 예산 완료', flush=True)
results, search_results = pd.DataFrame(rows), pd.DataFrame(search_rows)
assert len(results) == 600 and len(search_results) == 90
assert results.groupby(['policy','budget']).size().eq(30).all()
display(results.groupby(['policy','budget']).agg(hit=('hit','mean'), recall=('recall','mean'),
    all_evidence_questions=('all_evidence','sum'), median_tokens=('context_tokens','median')))
from IPython.display import JSON
display(JSON({'results':rows, 'search_results':search_rows}, expanded=False))


[1/30] S01 삼성전자 검색


  4개 정책 × 5개 예산 완료


[2/30] S02 삼성전자 검색


  4개 정책 × 5개 예산 완료


[3/30] S03 삼성전자 검색


...

  4개 정책 × 5개 예산 완료


[30/30] M_H HDC 검색


  4개 정책 × 5개 예산 완료


hit    recall  all_evidence_questions  \
policy       budget                                                  
A            16000      0.866667  0.833333                      24   
             2400       0.866667  0.833333                      24   
             32000      0.866667  0.833333                      24   
             8000       0.866667  0.833333                      24   
             unlimited  0.866667  0.833333                      24   
B            16000      0.833333  0.816667                      24   
             2400       0.766667  0.750000                      22   
             32000      0.833333  0.816667                      24   
             8000       0.833333  0.816667                      24   
             unlimited  0.833333  0.816667                      24   
C-full       16000      0.333333  0.333333                      10   
             2400       0.000000  0.000000                       0   
             32000      0.633333  0.633333                      19   
             8000       0.000000  0.000000                       0   
             unlimited  1.000000  1.000000                      30   
C-seed-first 16000      0.866667  0.850000                      25   
             2400       0.866667  0.850000                      25   
             32000      0.866667  0.850000                      25   
             8000       0.866667  0.850000                      25   
             unlimited  0.866667  0.850000                      25   

                        median_tokens  
policy       budget                    
A            16000             1691.0  
             2400              1691.0  
             32000             1691.0  
             8000              1691.0  
             unlimited         1691.0  
B            16000             4276.0  
             2400              2052.0  
             32000             4276.0  
             8000              4276.0  
             unlimited         4276.0  
C-full       16000             3617.0  
             2400                 0.0  
             32000            19544.5  
             8000                 0.0  
             unlimited        36889.0  
C-seed-first 16000            11242.0  
             2400              1941.0  
             32000            13257.0  
             8000              5173.5  
             unlimited        13257.0

<IPython.core.display.JSON object>

## 7. 결과와 개별 실패
무제한 결과와 예산 결과를 분리한다. Recall 차이가 없으면 작은 문맥이 운영 후보지만, 아직 LLM 답변 정확도는 평가하지 않았다.

In [7]:
summary = results.groupby(['policy','budget']).agg(
    questions=('id','count'), hits=('hit','sum'), hit=('hit','mean'), recall=('recall','mean'),
    all_evidence_questions=('all_evidence','sum'), median_tokens=('context_tokens','median'),
    max_tokens=('context_tokens','max'), median_context_ms=('context_ms','median'))
display(summary)
display(search_results.groupby('method')[['leaf_hit','leaf_mrr','leaf_recall','search_ms']].mean())
display(results.groupby(['dataset_group','policy','budget'])[['hit','recall','context_tokens']].mean())
for label in ('unlimited','8000'):
    print('질문별 근거 Recall:', label)
    display(results[results['budget'] == label].pivot(index=['company','id'], columns='policy', values='recall'))
with pd.option_context('display.max_rows', 150, 'display.max_colwidth', 160):
    display(pd.DataFrame(failure_rows))
print('C의 두 정책은 검색 순위가 같다. 차이는 문맥 반환과 예산 처리에서 발생한다.')


questions  hits       hit    recall  \
policy       budget                                           
A            16000             30    26  0.866667  0.833333   
             2400              30    26  0.866667  0.833333   
             32000             30    26  0.866667  0.833333   
             8000              30    26  0.866667  0.833333   
             unlimited         30    26  0.866667  0.833333   
B            16000             30    25  0.833333  0.816667   
             2400              30    23  0.766667  0.750000   
             32000             30    25  0.833333  0.816667   
             8000              30    25  0.833333  0.816667   
             unlimited         30    25  0.833333  0.816667   
C-full       16000             30    10  0.333333  0.333333   
             2400              30     0  0.000000  0.000000   
             32000             30    19  0.633333  0.633333   
             8000              30     0  0.000000  0.000000   
             unlimited         30    30  1.000000  1.000000   
C-seed-first 16000             30    26  0.866667  0.850000   
             2400              30    26  0.866667  0.850000   
             32000             30    26  0.866667  0.850000   
             8000              30    26  0.866667  0.850000   
             unlimited         30    26  0.866667  0.850000   

                        all_evidence_questions  median_tokens  max_tokens  \
policy       budget                                                         
A            16000                          24         1691.0        1886   
             2400                           24         1691.0        1886   
             32000                          24         1691.0        1886   
             8000                           24         1691.0        1886   
             unlimited                      24         1691.0        1886   
B            16000                          24         4276.0        5724   
             2400                           22         2052.0        2357   
             32000                          24         4276.0        5724   
             8000                           24         4276.0        5724   
             unlimited                      24         4276.0        5724   
C-full       16000                          10         3617.0       15429   
             2400                            0            0.0           0   
             32000                          19        19544.5       30378   
             8000                            0            0.0        7234   
             unlimited                      30        36889.0      908791   
C-seed-first 16000                          25        11242.0       15762   
             2400                           25         1941.0        2397   
             32000                          25        13257.0       23098   
             8000                           25         5173.5        7772   
             unlimited                      25        13257.0      379148   

                        median_context_ms  
policy       budget                        
A            16000               1.157432  
             2400                5.607742  
             32000               1.128393  
             8000                1.189034  
             unlimited           3.863494  
B            16000               1.703211  
             2400                7.406849  
             32000               1.687830  
             8000                2.356151  
             unlimited           6.622913  
C-full       16000               1.837107  
             2400                0.039928  
             32000               7.297104  
             8000                0.006711  
             unlimited          55.301139  
C-seed-first 16000               9.622089  
             2400               22.185926  
             32000               9.530266  
             8000               11.693674  
             unlim

,leaf_hit,leaf_mrr,leaf_recall,search_ms
method,,,,
heading_parent_child,0.866667,0.564444,0.833333,12.768538
parent_child,0.800000,0.574444,0.766667,13.063408
title_boundary,0.866667,0.547778,0.833333,13.062915


hit  recall  context_tokens
dataset_group policy       budget                                 
multi_5       A            16000      0.80    0.60         1734.60
                           2400       0.80    0.60         1734.60
                           32000      0.80    0.60         1734.60
                           8000       0.80    0.60         1734.60
                           unlimited  0.80    0.60         1734.60
              B            16000      0.80    0.70         4123.60
                           2400       0.80    0.70         1960.80
                           32000      0.80    0.70         4123.60
                           8000       0.80    0.70         4123.60
                           unlimited  0.80    0.70         4123.60
              C-full       16000      0.40    0.40         5032.20
                           2400       0.00    0.00            0.00
                           32000      0.60    0.60        13748.80
                           8000       0.00    0.00            0.00
                           unlimited  1.00    1.00       271009.40
              C-seed-first 16000      0.80    0.70        10657.20
                           2400       0.80    0.70         1988.20
                           32000      0.80    0.70        13171.40
                           8000       0.80    0.70         4311.80
                           unlimited  0.80    0.70        47832.40
single_25     A            16000      0.88    0.88         1669.04
                           2400       0.88    0.88         1669.04
                           32000      0.88    0.88         1669.04
                           8000       0.88    0.88         1669.04
                           unlimited  0.88    0.88         1669.04
              B            16000      0.84    0.84         4111.52
                           2400       0.76    0.76         1976.24
                           32000      0.84    0.84         4111.52
                           8000       0.84    0.84         4111.52
                           unlimited  0.84    0.84         4111.52
              C-full       16000      0.32    0.32         5953.56
                           2400       0.00    0.00            0.00
                           32000      0.64    0.64        16587.72
                           8000       0.00    0.00          289.36
                           unlimited  1.00    1.00       180589.04
              C-seed-first 16000      0.88    0.88        11091.36
                           2400       0.88    0.88         1956.04
                           32000      0.88    0.88        12997.16
                           8000       0.88    0.88         5236.40
                           unlimited  0.88    0.88        29988.56

질문별 근거 Recall: unlimited


policy         A    B  C-full  C-seed-first
company id                                 
HDC     H01  1.0  1.0     1.0           1.0
        H02  1.0  1.0     1.0           1.0
        H03  1.0  1.0     1.0           1.0
        H04  0.0  0.0     1.0           0.0
        H05  1.0  1.0     1.0           1.0
        M_H  0.0  0.0     1.0           0.0
LG생활건강  L01  0.0  0.0     1.0           0.0
        L02  1.0  1.0     1.0           1.0
        L03  1.0  1.0     1.0           1.0
        L04  1.0  1.0     1.0           1.0
        L05  1.0  1.0     1.0           1.0
        M_L  0.5  1.0     1.0           1.0
NAVER   M_N  1.0  1.0     1.0           1.0
        N01  1.0  1.0     1.0           1.0
        N02  1.0  1.0     1.0           1.0
        N03  1.0  1.0     1.0           1.0
        N04  1.0  1.0     1.0           1.0
        N05  1.0  1.0     1.0           1.0
리노공업    M_R  0.5  0.5     1.0           0.5
        R01  0.0  0.0     1.0           0.0
        R02  1.0  1.0     1.0           1.0
        R03  1.0  1.0     1.0           1.0
        R04  1.0  1.0     1.0           1.0
        R05  1.0  0.0     1.0           1.0
삼성전자    M_S  1.0  1.0     1.0           1.0
        S01  1.0  1.0     1.0           1.0
        S02  1.0  1.0     1.0           1.0
        S03  1.0  1.0     1.0           1.0
        S04  1.0  1.0     1.0           1.0
        S05  1.0  1.0     1.0           1.0

질문별 근거 Recall: 8000


policy         A    B  C-full  C-seed-first
company id                                 
HDC     H01  1.0  1.0     0.0           1.0
        H02  1.0  1.0     0.0           1.0
        H03  1.0  1.0     0.0           1.0
        H04  0.0  0.0     0.0           0.0
        H05  1.0  1.0     0.0           1.0
        M_H  0.0  0.0     0.0           0.0
LG생활건강  L01  0.0  0.0     0.0           0.0
        L02  1.0  1.0     0.0           1.0
        L03  1.0  1.0     0.0           1.0
        L04  1.0  1.0     0.0           1.0
        L05  1.0  1.0     0.0           1.0
        M_L  0.5  1.0     0.0           1.0
NAVER   M_N  1.0  1.0     0.0           1.0
        N01  1.0  1.0     0.0           1.0
        N02  1.0  1.0     0.0           1.0
        N03  1.0  1.0     0.0           1.0
        N04  1.0  1.0     0.0           1.0
        N05  1.0  1.0     0.0           1.0
리노공업    M_R  0.5  0.5     0.0           0.5
        R01  0.0  0.0     0.0           0.0
        R02  1.0  1.0     0.0           1.0
        R03  1.0  1.0     0.0           1.0
        R04  1.0  1.0     0.0           1.0
        R05  1.0  0.0     0.0           1.0
삼성전자    M_S  1.0  1.0     0.0           1.0
        S01  1.0  1.0     0.0           1.0
        S02  1.0  1.0     0.0           1.0
        S03  1.0  1.0     0.0           1.0
        S04  1.0  1.0     0.0           1.0
        S05  1.0  1.0     0.0           1.0

,id,company,policy,budget,question,missing_ids,context_tokens,first_context_preview
0,S01,삼성전자,C-full,8000,삼성전자가 주식을 일반에 공개한 시점은 언제인가?,[S01],0,
1,S02,삼성전자,C-full,8000,삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?,[S02],0,
2,S03,삼성전자,C-full,8000,삼성전자 본사의 소재지는 어디인가?,[S03],0,
3,S04,삼성전자,C-full,8000,삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?,[S04],0,
4,S05,삼성전자,C-full,8000,삼성전자 전장 사업에 해당하는 하만의 주요 제품은?,[S05],0,
5,R01,리노공업,A,unlimited,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01],1690,"아울러 반도체 시장의 변화에 따른 원가 절감 및 성능 향상을 위한 공정개선, 즉 반도체 패키지 공정 축소 및 개선에 대한 기술변화를 선지하여 이에 적합한 제품을 출시할 수 있도록 연구개발 인력을 투입하고 있으며, 신규산업분야에 진출하기 위한 제품 및 공정연구도 함께 하고 있습니다..."
6,R01,리노공업,A,8000,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01],1690,"아울러 반도체 시장의 변화에 따른 원가 절감 및 성능 향상을 위한 공정개선, 즉 반도체 패키지 공정 축소 및 개선에 대한 기술변화를 선지하여 이에 적합한 제품을 출시할 수 있도록 연구개발 인력을 투입하고 있으며, 신규산업분야에 진출하기 위한 제품 및 공정연구도 함께 하고 있습니다..."
7,R01,리노공업,B,unlimited,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01],3696,"|\n상품화 내용 - 시제품 검증을 마치고 유효성이 확인되었으며, High Power 대응 Lid 개발로 시장에서의 경쟁력 확보 |\n연구과제 | 5G 대응 AIP/OTA TEST 관련 테스트 소켓 개발 |\n연구기관 | 리노공업 기술연구소 |\n연구결과 및 기대효과 - 5G 대..."
8,R01,리노공업,B,8000,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01],3696,"|\n상품화 내용 - 시제품 검증을 마치고 유효성이 확인되었으며, High Power 대응 Lid 개발로 시장에서의 경쟁력 확보 |\n연구과제 | 5G 대응 AIP/OTA TEST 관련 테스트 소켓 개발 |\n연구기관 | 리노공업 기술연구소 |\n연구결과 및 기대효과 - 5G 대..."
9,R01,리노공업,C-full,8000,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01],0,


C의 두 정책은 검색 순위가 같다. 차이는 문맥 반환과 예산 처리에서 발생한다.


## 8. 실행·속도·자원 기록
GPU 수치는 이 Python 프로세스의 PyTorch tensor 할당·예약 peak다. 다른 프로세스를 포함한 GPU 전체 사용량 또는 Qwen 메모리는 아니다.

In [8]:
torch.cuda.synchronize(DEVICE)
runtime = {'observed_at_utc':datetime.now(timezone.utc).isoformat(), 'dataset_type':'development',
    'questions':len(qcases), 'companies':len(REPORTS), 'reports':len(REPORTS),
    'evaluation_sha256':evaluation_sha256, 'source30_code_sha256':source30_sha,
    'source_xml_sha256':source_sha256, 'model':MODEL_ID, 'model_revision':MODEL_REVISION,
    'device':str(next(model.parameters()).device), 'gpu':torch.cuda.get_device_name(0), 'dtype':'float32',
    'gpu_initial_free_MiB':gpu_initial_free_MiB,
    'torch_peak_allocated_MiB':torch.cuda.max_memory_allocated(DEVICE)/1024**2,
    'torch_peak_reserved_MiB':torch.cuda.max_memory_reserved(DEVICE)/1024**2,
    'model_parameter_MiB':parameter_MiB, 'model_load_seconds':model_load_seconds,
    'policies':list(POLICIES), 'budgets_e5_tokens':list(BUDGETS),
    'vector_bm25_weights':[0.5,0.5], 'candidate_depth_each':20, 'rrf_k':60, 'top_seeds':5,
    'llm_calls':0, 'openai_embedding_calls':0,
    'new_embedding_texts':sum(r['new_embedding_texts'] for r in e5_rows),
    'e5_build_seconds':sum(r['e5_build_seconds'] for r in e5_rows),
    'packages':{p:importlib.metadata.version(p) for p in ['llama-index-core','transformers','torch','kiwipiepy']}}
display(JSON(runtime, expanded=False))
print(json.dumps(runtime, ensure_ascii=False, indent=2))
print('외부 LLM/임베딩 요금 없음. 전력·CPU RAM·실제 답변 생성 비용/정확도는 미측정.')
for connection in bm25_connections.values():
    connection.close()
print('완료: 결과 저장 후 사용자와 채택 여부를 결정한다. 운영 pipeline은 변경하지 않는다.')


<IPython.core.display.JSON object>

{
  "observed_at_utc": "2026-10-05T17:08:55.478960+00:00",
  "dataset_type": "development",
  "questions": 30,
  "companies": 5,
  "reports": 5,
  "evaluation_sha256": "a8bbc6202b3b0956c7519660a6e760eb1dab9be1d808118bdba1e7557c3c6ebe",
  "source30_code_sha256": "d38daeb5664a0ba916fa908187de7abf39dbf57311771e08b2005065ffc6d952",
  "source_xml_sha256": {
    "삼성전자": "b0682fd46de50ecc41f074bb43cb58cf574dd1750343710f169c58ab5066ad2f",
    "리노공업": "a23af2dbcc2f4de4be59c5925ed45fceae84c23f4e78ff3365071f5d6cdbeb93",
    "NAVER": "b1478f612c3ff07373d14fe9c7638936c77fcea2130aa4ec6202bef90bf3bfa9",
    "LG생활건강": "9ccc7d18e1ba66ce3982f41899d22d1dcb7ae8dfaf784f21a9b176fa294cdb16",
    "HDC": "53b41f2833b68f06bfea13a07abed1a7cdb3caf53c44e615cb21a4c736411e9f"
  },
  "model": "intfloat/multilingual-e5-small",
  "model_revision": "5697a65b0a002a92fe8c4fc9d495303ffff9c7d2",
  "device": "cuda:0",
  "gpu": "NVIDIA GeForce RTX 3090",
  "dtype": "float32",
  "gpu_initial_free_MiB": 16038.6875,
  "torch_pea